In [5]:
from torch.nn import CrossEntropyLoss, MSELoss, BCEWithLogitsLoss


In [6]:
from typing import List, Optional, Tuple, Union

In [7]:
import torch

In [8]:
from transformers import RobertaPreTrainedModel, RobertaModel

In [11]:
from torch.autograd import Function

In [9]:
from transformers.models.roberta.modeling_roberta import RobertaClassificationHead, SequenceClassifierOutput

In [12]:
class GradReverse(Function):
    @staticmethod
    def forward(ctx, x):
        return x.view_as(x)

    @staticmethod
    def backward(ctx, grad_output):
        return grad_output.neg()


def grad_reverse(x):
    return GradReverse.apply(x)


class RobertaForSequenceClassificationGradientReverse(RobertaPreTrainedModel):
    def __init__(self, config):
        super().__init__(config)
        self.num_labels = config.num_labels
        self.config = config

        self.roberta = RobertaModel(config, add_pooling_layer=False)
        self.classifier = RobertaClassificationHead(config)
        self.classifierDomain = RobertaClassificationHead(config)

        # Initialize weights and apply final processing
        self.post_init()


    def forward(
        self,
        input_ids: Optional[torch.LongTensor] = None,
        attention_mask: Optional[torch.FloatTensor] = None,
        token_type_ids: Optional[torch.LongTensor] = None,
        position_ids: Optional[torch.LongTensor] = None,
        head_mask: Optional[torch.FloatTensor] = None,
        inputs_embeds: Optional[torch.FloatTensor] = None,
        labels: Optional[torch.LongTensor] = None,
        output_attentions: Optional[bool] = None,
        output_hidden_states: Optional[bool] = None,
        return_dict: Optional[bool] = None,
        domain_label: Optional[str] = None,
    ) -> Union[dict, Tuple[torch.Tensor], SequenceClassifierOutput]:
        r"""
        labels (`torch.LongTensor` of shape `(batch_size,)`, *optional*):
            Labels for computing the sequence classification/regression loss. Indices should be in `[0, ...,
            config.num_labels - 1]`. If `config.num_labels == 1` a regression loss is computed (Mean-Square loss), If
            `config.num_labels > 1` a classification loss is computed (Cross-Entropy).
        """
        return_dict = return_dict if return_dict is not None else self.config.use_return_dict

        outputs = self.roberta(
            input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
            position_ids=position_ids,
            head_mask=head_mask,
            inputs_embeds=inputs_embeds,
            output_attentions=output_attentions,
            output_hidden_states=output_hidden_states,
            return_dict=return_dict,
        )
        sequence_output = outputs[0]
        logits = self.classifier(sequence_output)
        logitsDomain = self.classifierDomain(grad_reverse(sequence_output))

        loss = None
        if labels is not None:
            # move labels to correct device to enable model parallelism
            labels = labels.to(logits.device)
            if self.config.problem_type is None:
                if self.num_labels == 1:
                    self.config.problem_type = "regression"
                elif self.num_labels > 1 and (labels.dtype == torch.long or labels.dtype == torch.int):
                    self.config.problem_type = "single_label_classification"
                else:
                    self.config.problem_type = "multi_label_classification"

            if self.config.problem_type == "regression":
                loss_fct = ()
                if self.num_labels == 1:
                    loss = loss_fct(logits.squeeze(), labels.squeeze())
                else:
                    loss = loss_fct(logits, labels)
            elif self.config.problem_type == "single_label_classification":
                loss_fct = CrossEntropyLoss()
                loss = loss_fct(logits.view(-1, self.num_labels), labels.view(-1))
            elif self.config.problem_type == "multi_label_classification":
                loss_fct = BCEWithLogitsLoss()
                loss = loss_fct(logits, labels)

        
        
        
        lossDomain = loss_fct(logitsDomain, domain_label)
        
        return {"lossMain":loss, "lossDomain":lossDomain}
        # if not return_dict:
        #     output = (logits,) + outputs[2:]
        #     return ((loss,) + output) if loss is not None else output

        
        
#         ## Add MixUp
#         sequence_output_stack = []
#         labels_mixup_stack = []
#         for k,v in self.n_aug_targets.items():
#             if (v > 0) and (self.n_aug_tracker[k] < v):
                
#                 indexMixUp = (labels == aug_query_dict[k]['labels'] ) & torch.tensor((np.array(domain_label.cpu()) == aug_query_dict[k]['domain_idx'])).to(globalconfig.device)
                
#                 sequence_output_forMixUp = sequence_output[indexMixUp]
                
#                 all_combos = list(itertools.combinations(range(sequence_output_forMixUp.shape[0]), 2))
                
#                 _ct = 0
#                 for _combos in all_combos:
                    
                    
#                     _lambda = np.random.beta(globalconfig.alpha, globalconfig.alpha)
                    
#                     sequence_output_stack.append( _lambda * sequence_output_forMixUp[_combos[0]] + (1-_lambda) * sequence_output_forMixUp[_combos[1]])
#                     _ct += 1
                    
#                     self.n_aug_tracker[k] += 1
                    
#                     if self.n_aug_tracker[k] == v:
#                         break
                        
#                 labels_mixup_stack.append([aug_query_dict[k]['labels']] * _ct)

        
#         if labels is not None:
#             if len(sequence_output_stack) > 0:
#                 sequence_output_mixup = torch.stack(sequence_output_stack)

#                 logits_mixup = self.classifier(sequence_output_mixup)
                
#                 breakpoint()

#                 labels_mixup = torch.tensor(np.hstack(labels_mixup_stack), dtype=torch.int64).to(globalconfig.device)

#                 loss_mixup = loss_fct(logits_mixup.view(-1, self.num_labels), labels_mixup.view(-1))

#                 loss_combine = loss + loss_mixup
                

#             else:
#                 loss_combine = loss
#         else:
#             loss_combine = None
                    
        
        # return SequenceClassifierOutput(
        #     loss=loss+lossDomain,
        #     logits=logits,
        #     hidden_states=outputs.hidden_states,
        #     attentions=outputs.attentions,
        # )
    
    
    
    def inference(
        self,
        input_ids: Optional[torch.LongTensor] = None,
        attention_mask: Optional[torch.FloatTensor] = None,
        token_type_ids: Optional[torch.LongTensor] = None,
        position_ids: Optional[torch.LongTensor] = None,
        head_mask: Optional[torch.FloatTensor] = None,
        inputs_embeds: Optional[torch.FloatTensor] = None,
        labels: Optional[torch.LongTensor] = None,
        output_attentions: Optional[bool] = None,
        output_hidden_states: Optional[bool] = None,
        return_dict: Optional[bool] = None,
    ) -> Union[Tuple[torch.Tensor], SequenceClassifierOutput]:
        r"""
        labels (`torch.LongTensor` of shape `(batch_size,)`, *optional*):
            Labels for computing the sequence classification/regression loss. Indices should be in `[0, ...,
            config.num_labels - 1]`. If `config.num_labels == 1` a regression loss is computed (Mean-Square loss), If
            `config.num_labels > 1` a classification loss is computed (Cross-Entropy).
        """
        return_dict = return_dict if return_dict is not None else self.config.use_return_dict

        outputs = self.roberta(
            input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
            position_ids=position_ids,
            head_mask=head_mask,
            inputs_embeds=inputs_embeds,
            output_attentions=output_attentions,
            output_hidden_states=output_hidden_states,
            return_dict=return_dict,
        )
        sequence_output = outputs[0]
        logits = self.classifier(sequence_output)

        loss = None
        if labels is not None:
            # move labels to correct device to enable model parallelism
            labels = labels.to(logits.device)
            if self.config.problem_type is None:
                if self.num_labels == 1:
                    self.config.problem_type = "regression"
                elif self.num_labels > 1 and (labels.dtype == torch.long or labels.dtype == torch.int):
                    self.config.problem_type = "single_label_classification"
                else:
                    self.config.problem_type = "multi_label_classification"

            if self.config.problem_type == "regression":
                loss_fct = MSELoss()
                if self.num_labels == 1:
                    loss = loss_fct(logits.squeeze(), labels.squeeze())
                else:
                    loss = loss_fct(logits, labels)
            elif self.config.problem_type == "single_label_classification":
                loss_fct = CrossEntropyLoss()
                loss = loss_fct(logits.view(-1, self.num_labels), labels.view(-1))
            elif self.config.problem_type == "multi_label_classification":
                loss_fct = BCEWithLogitsLoss()
                loss = loss_fct(logits, labels)

        if not return_dict:
            output = (logits,) + outputs[2:]
            return ((loss,) + output) if loss is not None else output

        return SequenceClassifierOutput(
            loss=loss,
            logits=logits,
            hidden_states=outputs.hidden_states,
            attentions=outputs.attentions,
        )


In [13]:
model = RobertaForSequenceClassificationGradientReverse.from_pretrained("roberta-base", num_labels=2)

/home/NETID/xiruod/anaconda3/envs/llama2/lib/python3.9/site-packages/huggingface_hub/file_download.py:1150: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of RobertaForSequenceClassificationGradientReverse were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifierDomain.out_proj.bias', 'classifierDomain.dense.bias', 'classifierDomain.dense.weight', 'classifier.dense.bias', 'classifier.out_proj.weight', 'classifierDomain.out_proj.weight', 'classifier.dense.weight', 'classifier.out_proj.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [14]:
model

RobertaForSequenceClassificationGradientReverse(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True

# Test Loading

In [6]:
from transformers import (
    Trainer,
    TrainingArguments,
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainerCallback,
    default_data_collator,
)
import torch

In [3]:
model = AutoModelForSequenceClassification.from_pretrained(
    "roberta-base",
    use_safetensors=False,
)

/home/NETID/xiruod/anaconda3/envs/llama2/lib/python3.9/site-packages/huggingface_hub/file_download.py:1150: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.weight', 'classifier.out_proj.weight', 'classifier.dense.bias', 'classifier.out_proj.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [11]:
wtfile = "/bime-munin/xiruod/MMD/roberta-base_CD/n200/set-566-epoch100/pytorch_model.bin"

In [17]:
wtfile = "/bime-munin/xiruod/GradientReverse/roberta-base_CD/n200/set-566-epoch6/pytorch_model.bin"

In [16]:
!ls /bime-munin/xiruod/GradientReverse/roberta-base_CD/n200/

set-566-epoch50  set-566-epoch6


In [25]:
tmp = torch.load(
        wtfile,
        map_location="cpu",
        # map_location=lambda storage, loc: storage,
    )

key_ToRemove = ["classifierDomain.dense.weight", "classifierDomain.dense.bias", "classifierDomain.out_proj.weight", "classifierDomain.out_proj.bias"]


for _ in key_ToRemove:
    del tmp[_]

# this step cannot be ignored here...
model.load_state_dict(
    tmp
)